In [67]:
import pandas as pd
import numpy as np
import re
import numpy as np

In [68]:
df = pd.read_csv('quikr.csv')
df.head()

,lozad src,car-info__footer,prime-features,price
0,https://teja8.kuikr.com/i5/20260528/White-2024...,Hyundai Grand i10 NIOS Asta Petrol - 2024,"3,381 kms / Petrol / 1st Owner","₹8,50,000"
1,https://teja8.kuikr.com/i5/20260528/White-2015...,Maruti Suzuki Swift LDi - 2015,"87,000 kms / Diesel","₹3,80,000"
2,https://teja8.kuikr.com/i6/20260509/Red-2011-H...,Hyundai Santro Xing GLS - 2011,"84,967 kms / Petrol","₹1,85,000"
3,https://teja8.kuikr.com/i5/20260527/Black-2021...,Honda WR V - 2021,"82,000 kms / Petrol / 1st Owner","₹6,25,000"
4,https://teja8.kuikr.com/i4/20260526/Black-2024...,Maruti Suzuki Maruti Suzuki BREZZA BREZZA ZXI ...,"12,500 kms / Hybrid,Hybrid / one,one","₹12,90,000"


In [69]:
df.shape

(1701, 4)

In [70]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1701 entries, 0 to 1700
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   lozad src         911 non-null    object
 1   car-info__footer  1701 non-null   object
 2   prime-features    1701 non-null   object
 3   price             1701 non-null   object
dtypes: object(4)
memory usage: 53.3+ KB


## Quality
Separte car-info_footer into car_name and company_name column

separte prime feature column into year and kms_driven column

price data type is object

fuel_type has null values

price has ask for price data value

kms_driven object into int

keep 3 words of name



In [71]:
def extract_car_info(car_info_string):
    # Regex to find a 4-digit year, possibly preceded by a hyphen or space
    year_match = re.search(r'\b(\d{4})\b', car_info_string)
    year = year_match.group(1) if year_match else None

    car_name = car_info_string
    if year:
        # Remove the year and any preceding hyphen/space from the car name string
        car_name = re.sub(r'(?:\s*-)?\s*' + re.escape(year) + r'\b', '', car_info_string).strip()

    # Further clean up the car_name by removing common extraneous information like 'Petrol', 'Diesel', 'Hybrid'
    # This part can be refined based on specific needs if other details are to be kept or removed
    car_name = re.sub(r'\s*(Petrol|Diesel|Hybrid|CNG)\s*', ' ', car_name, flags=re.IGNORECASE).strip()
    car_name = re.sub(r'\s*\d+(?:st|nd|rd|th) Owner\s*', '', car_name, flags=re.IGNORECASE).strip()
    car_name = re.sub(r'\s*\d+,?\d*\s*kms\s*', '', car_name, flags=re.IGNORECASE).strip()


    return car_name, year

# Apply the function to the 'car-info__footer' column
df[['car_name', 'year']] = df['car-info__footer'].apply(lambda x: pd.Series(extract_car_info(x)))

# Convert 'year' column to numeric type, coercing errors to NaN
df['year'] = pd.to_numeric(df['year'], errors='coerce')

# Display the updated DataFrame with the new columns
display(df.head())

,lozad src,car-info__footer,prime-features,price,car_name,year
0,https://teja8.kuikr.com/i5/20260528/White-2024...,Hyundai Grand i10 NIOS Asta Petrol - 2024,"3,381 kms / Petrol / 1st Owner","₹8,50,000",Hyundai Grand i10 NIOS Asta,2024
1,https://teja8.kuikr.com/i5/20260528/White-2015...,Maruti Suzuki Swift LDi - 2015,"87,000 kms / Diesel","₹3,80,000",Maruti Suzuki Swift LDi,2015
2,https://teja8.kuikr.com/i6/20260509/Red-2011-H...,Hyundai Santro Xing GLS - 2011,"84,967 kms / Petrol","₹1,85,000",Hyundai Santro Xing GLS,2011
3,https://teja8.kuikr.com/i5/20260527/Black-2021...,Honda WR V - 2021,"82,000 kms / Petrol / 1st Owner","₹6,25,000",Honda WR V,2021
4,https://teja8.kuikr.com/i4/20260526/Black-2024...,Maruti Suzuki Maruti Suzuki BREZZA BREZZA ZXI ...,"12,500 kms / Hybrid,Hybrid / one,one","₹12,90,000",Maruti Suzuki Maruti Suzuki BREZZA BREZZA ZXI ZXI,2024


In [72]:
def extract_prime_features(features_string):
    kms_driven = None
    fuel_type = None

    # Extract kms driven (e.g., '3,381 kms', '87,000 kms')
    kms_match = re.search(r'(\d{1,3}(?:,\d{3})*)\s*kms', features_string, re.IGNORECASE)
    if kms_match:
        kms_driven = kms_match.group(1).replace(',', '')

    # Extract fuel type (e.g., 'Petrol', 'Diesel', 'Hybrid', 'CNG')
    fuel_match = re.search(r'\b(Petrol|Diesel|Hybrid|CNG)\b', features_string, re.IGNORECASE)
    if fuel_match:
        fuel_type = fuel_match.group(1)

    return kms_driven, fuel_type

# Apply the function to the 'prime-features' column
df[['kms_driven', 'fuel_type']] = df['prime-features'].apply(lambda x: pd.Series(extract_prime_features(x)))

# Display the updated DataFrame with the new columns
display(df.head())

,lozad src,car-info__footer,prime-features,price,car_name,year,kms_driven,fuel_type
0,https://teja8.kuikr.com/i5/20260528/White-2024...,Hyundai Grand i10 NIOS Asta Petrol - 2024,"3,381 kms / Petrol / 1st Owner","₹8,50,000",Hyundai Grand i10 NIOS Asta,2024,3381,Petrol
1,https://teja8.kuikr.com/i5/20260528/White-2015...,Maruti Suzuki Swift LDi - 2015,"87,000 kms / Diesel","₹3,80,000",Maruti Suzuki Swift LDi,2015,87000,Diesel
2,https://teja8.kuikr.com/i6/20260509/Red-2011-H...,Hyundai Santro Xing GLS - 2011,"84,967 kms / Petrol","₹1,85,000",Hyundai Santro Xing GLS,2011,84967,Petrol
3,https://teja8.kuikr.com/i5/20260527/Black-2021...,Honda WR V - 2021,"82,000 kms / Petrol / 1st Owner","₹6,25,000",Honda WR V,2021,82000,Petrol
4,https://teja8.kuikr.com/i4/20260526/Black-2024...,Maruti Suzuki Maruti Suzuki BREZZA BREZZA ZXI ...,"12,500 kms / Hybrid,Hybrid / one,one","₹12,90,000",Maruti Suzuki Maruti Suzuki BREZZA BREZZA ZXI ZXI,2024,12500,Hybrid


In [73]:
df['company_name'] = df['car_name'].apply(lambda x: x.split(' ')[0] if isinstance(x, str) else None)
display(df.head())

,lozad src,car-info__footer,prime-features,price,car_name,year,kms_driven,fuel_type,company_name
0,https://teja8.kuikr.com/i5/20260528/White-2024...,Hyundai Grand i10 NIOS Asta Petrol - 2024,"3,381 kms / Petrol / 1st Owner","₹8,50,000",Hyundai Grand i10 NIOS Asta,2024,3381,Petrol,Hyundai
1,https://teja8.kuikr.com/i5/20260528/White-2015...,Maruti Suzuki Swift LDi - 2015,"87,000 kms / Diesel","₹3,80,000",Maruti Suzuki Swift LDi,2015,87000,Diesel,Maruti
2,https://teja8.kuikr.com/i6/20260509/Red-2011-H...,Hyundai Santro Xing GLS - 2011,"84,967 kms / Petrol","₹1,85,000",Hyundai Santro Xing GLS,2011,84967,Petrol,Hyundai
3,https://teja8.kuikr.com/i5/20260527/Black-2021...,Honda WR V - 2021,"82,000 kms / Petrol / 1st Owner","₹6,25,000",Honda WR V,2021,82000,Petrol,Honda
4,https://teja8.kuikr.com/i4/20260526/Black-2024...,Maruti Suzuki Maruti Suzuki BREZZA BREZZA ZXI ...,"12,500 kms / Hybrid,Hybrid / one,one","₹12,90,000",Maruti Suzuki Maruti Suzuki BREZZA BREZZA ZXI ZXI,2024,12500,Hybrid,Maruti


In [74]:
df = df[['car_name', 'company_name', 'year', 'price', 'kms_driven', 'fuel_type']]
df.head()

,car_name,company_name,year,price,kms_driven,fuel_type
0,Hyundai Grand i10 NIOS Asta,Hyundai,2024,"₹8,50,000",3381,Petrol
1,Maruti Suzuki Swift LDi,Maruti,2015,"₹3,80,000",87000,Diesel
2,Hyundai Santro Xing GLS,Hyundai,2011,"₹1,85,000",84967,Petrol
3,Honda WR V,Honda,2021,"₹6,25,000",82000,Petrol
4,Maruti Suzuki Maruti Suzuki BREZZA BREZZA ZXI ZXI,Maruti,2024,"₹12,90,000",12500,Hybrid


In [75]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1701 entries, 0 to 1700
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   car_name      1701 non-null   object
 1   company_name  1701 non-null   object
 2   year          1701 non-null   int64 
 3   price         1701 non-null   object
 4   kms_driven    1701 non-null   object
 5   fuel_type     1668 non-null   object
dtypes: int64(1), object(5)
memory usage: 79.9+ KB


## Cleaning

In [76]:
backup = df.copy()

In [77]:
df['year'] = df['year'].astype(int)
df['kms_driven'] = df['kms_driven'].astype(int)

In [78]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1701 entries, 0 to 1700
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   car_name      1701 non-null   object
 1   company_name  1701 non-null   object
 2   year          1701 non-null   int64 
 3   price         1701 non-null   object
 4   kms_driven    1701 non-null   int64 
 5   fuel_type     1668 non-null   object
dtypes: int64(2), object(4)
memory usage: 79.9+ KB


In [79]:
df['price'].unique()

array(['₹8,50,000', '₹3,80,000', '₹1,85,000', '₹6,25,000', '₹12,90,000',
       '₹3,25,000', '₹4,75,000', '₹8,65,000', '₹2,75,000', '₹6,10,000',
       '₹13,99,999', '₹14,00,000', '₹10,49,000', '₹4,45,000', '₹3,50,000',
       '₹4,99,999', '₹16,95,000', '₹4,99,000', '₹20,75,000', '₹14,50,000',
       '₹21,00,000', '₹16,50,000', '₹9,50,000', '₹4,41,000', '₹2,85,000',
       '₹2,29,999', '₹25,25,000', '₹11,90,000', '₹3,95,000', '₹8,00,000',
       '₹8,25,000', '₹4,90,000', '₹7,25,000', '₹34,50,000', '₹9,25,000',
       '₹17,55,000', '₹12,75,000', '₹5,95,000', '₹11,00,000',
       '₹45,50,000', '₹25,35,000', '₹12,30,000', '₹18,50,000',
       '₹13,95,000', '₹22,50,000', '₹6,50,000', '₹9,35,000', '₹2,65,000',
       '₹4,35,000', '₹14,25,000', '₹34,99,000', '₹27,50,000',
       '₹13,99,000', '₹22,75,000', '₹6,45,000', '₹33,00,000',
       '₹11,75,000', '₹15,45,000', '₹24,90,000', '₹9,85,000',
       '₹14,95,000', '₹4,25,000', '₹15,25,000', '₹21,75,000', '₹1,10,000',
       '₹18,40,000', '₹8

In [80]:
df = df[df['price'] != 'Ask For Price']
df['price'] = df['price'].str.replace('₹', '')
df['price'] = df['price'].str.replace(',', '').astype(int)
df['price']

,price
0,850000
1,380000
2,185000
3,625000
4,1290000
...,...
1696,12000
1697,450000
1698,2750000
1699,210000


In [81]:
#df['fuel_type'] = df['fuel_type'].dropna().reset_index(drop=True)
#df['fuel_type']

In [82]:
df['car_name'] = df['car_name'].apply(lambda x: ' '.join(x.split(' ')[:3]))
df['car_name']

,car_name
0,Hyundai Grand i10
1,Maruti Suzuki Swift
2,Hyundai Santro Xing
3,Honda WR V
4,Maruti Suzuki Maruti
...,...
1696,Mini Cooper S
1697,Hyundai Elite i20
1698,Toyota Innova Crysta
1699,Maruti Suzuki Wagon


In [83]:
df = df.dropna(subset=['fuel_type'])
df['fuel_type']

,fuel_type
0,Petrol
1,Diesel
2,Petrol
3,Petrol
4,Hybrid
...,...
1695,Petrol
1697,Petrol
1698,Diesel
1699,Petrol


In [84]:
df = df.reset_index(drop=True)

In [85]:
df.describe()

,year,price,kms_driven
count,1653.000000,1.653000e+03,1653.000000
mean,2016.621295,8.597161e+08,46433.570478
std,5.056038,2.434342e+10,28637.099252
min,1996.000000,3.000000e+00,0.000000
25%,2013.000000,2.650000e+05,22000.000000
50%,2017.000000,4.800000e+05,47000.000000
75%,2021.000000,8.500000e+05,70000.000000
max,2024.000000,7.000005e+11,99999.000000


In [86]:
df = df[df['price'] < 8e6].reset_index(drop=True)
df

,car_name,company_name,year,price,kms_driven,fuel_type
0,Hyundai Grand i10,Hyundai,2024,850000,3381,Petrol
1,Maruti Suzuki Swift,Maruti,2015,380000,87000,Diesel
2,Hyundai Santro Xing,Hyundai,2011,185000,84967,Petrol
3,Honda WR V,Honda,2021,625000,82000,Petrol
4,Maruti Suzuki Maruti,Maruti,2024,1290000,12500,Hybrid
...,...,...,...,...,...,...
1644,Honda Amaze 1.2,Honda,2015,250000,42000,Petrol
1645,Hyundai Elite i20,Hyundai,2018,450000,122,Petrol
1646,Toyota Innova Crysta,Toyota,2022,2750000,27000,Diesel
1647,Maruti Suzuki Wagon,Maruti,2014,210000,38400,Petrol


In [87]:
df.to_csv('Cleaned_car.csv')

## Model

In [88]:
x = df.drop(columns='price')
y = df['price']

In [89]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2)

In [90]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline

In [91]:
ohe = OneHotEncoder()
ohe.fit(x[['car_name', 'company_name', 'fuel_type']])

OneHotEncoder()

In [92]:
ohe.categories_

[array(['Audi A3 35', 'Audi A4 30', 'Audi A6 2.0', 'Audi Q3 35',
        'Audi Q5 30', 'Audi Q7 3.0', 'Audi Q7 45', 'BMW 1 Series',
        'BMW 3 Series', 'BMW 5 Series', 'BMW X1 sDrive20d',
        'BMW X1 xDrive20d', 'BMW X3 XDRIVE', 'Chevrolet Beat',
        'Chevrolet Beat LS', 'Chevrolet Beat LT', 'Chevrolet Cruze LTZ',
        'Chevrolet Sail 1.2', 'Chevrolet Spark LT', 'Chevrolet Tavera B2',
        'Datsun GO T', 'Datsun Redi GO', 'Datsun RediGo A',
        'Datsun RediGo S', 'Fiat Linea Dynamic', 'Fiat Punto Sport',
        'Force Motors Gurkha', 'Ford EcoSport Titanium',
        'Ford Endeavour 3.0L', 'Ford Fiesta 1.6', 'Ford Fiesta Classic',
        'Ford Figo Aspire', 'Ford Figo Duratec', 'Ford Figo EXI',
        'Ford Figo Titanium', 'Ford Figo Trend', 'Ford Figo ZXI',
        'Honda Amaze 1.2', 'Honda Amaze 1.5', 'Honda Brio 1.2',
        'Honda City 1.5', 'Honda City V', 'Honda City VX', 'Honda City ZX',
        'Honda Civic 1.8', 'Honda Civic 1.8V', 'Honda ELEVATE',
  

In [93]:
column_trans = make_column_transformer((OneHotEncoder(categories=ohe.categories_), ['car_name', 'company_name', 'fuel_type']),
                                       remainder='passthrough')

In [94]:
lr = LinearRegression()

In [95]:
pipe = make_pipeline(column_trans, lr)
pipe.fit(x_train, y_train)
y_pred = pipe.predict(x_test)
y_pred

array([ 1.37378794e+06,  7.09580847e+05,  2.40299859e+06,  1.68193500e+06,
        7.95181312e+05,  8.56258920e+05,  3.50026343e+05,  7.27282908e+05,
        1.61399328e+05,  6.78695633e+05,  2.17755737e+04,  3.39014955e+05,
        6.19060732e+05,  5.02466136e+05,  1.74227129e+06,  4.21264197e+05,
        2.77268942e+05,  1.00162302e+06,  8.65182157e+05,  3.73442593e+04,
        1.33929910e+06,  1.27490025e+06,  8.59711895e+05,  1.85015320e+05,
        1.06480729e+06,  1.20829300e+05,  4.36481118e+05,  1.63753298e+06,
        5.33291348e+05,  2.40664992e+05,  5.84465981e+05,  2.00246941e+06,
        4.52526809e+05,  5.99740237e+05,  4.26428867e+06,  1.39705520e+06,
       -3.42902447e+03,  3.47441629e+05,  1.89068398e+05,  2.66012539e+05,
        4.38589150e+05,  3.26780741e+05,  7.21332994e+05,  1.35503529e+04,
        4.28289597e+05,  1.69716533e+06,  1.33565437e+06,  2.00246941e+06,
        8.74096281e+05,  5.45135151e+05,  7.16815891e+05,  4.35820663e+05,
        1.63491114e+05,  

In [96]:
r2_score(y_test, y_pred)

0.6886779179860316

In [66]:
#checking which random_state has maximum r2_score
score = []
for i in range(1000):
    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=i)
    lr = LinearRegression()
    pipe = make_pipeline(column_trans, lr)
    pipe.fit(x_train, y_train)
    y_pred = pipe.predict(x_test)
    score.append(r2_score(y_test, y_pred))

In [97]:
np.argmax(score)

np.int64(984)

In [98]:
score[np.argmax(score)]

0.8366358135762981

In [100]:
#training on random_state with highest r2_score score
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = np.argmax(score))
lr = LinearRegression()
pipe = make_pipeline(column_trans, lr)
pipe.fit(x_train, y_train)
y_pred = pipe.predict(x_test)
r2_score(y_test, y_pred)

0.8366358135762981

In [101]:
import pickle
pickle.dump(pipe, open('LinearRegressionModel.pkl', 'wb'))

In [102]:
#testing prediction
pipe.predict(pd.DataFrame([['Maruti Suzuki Swift', 'Maruti', 2019, 100, 'Petrol']], columns=['car_name', 'company_name', 'year', 'kms_driven', 'fuel_type']))

array([587078.42913103])